# 02 - Month of Year Seasonality

Is October always good? Is September always bad?

Crypto folklore says:

- '**Sell in May and go away**' - summer is weak
- '**Uptober**' - October is bullish
- '**Rektember**' - September is bearish

**We test these claims with real data.**

Sections:

1. Mean return by calendar month
2. Volatility by month
3. Year × month heatmap
4. Quarter analysis (Q1/Q2/Q3/Q4)
5. Statistical test
6. Verdict: real pattern or folklore?

## Setup

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

pd.set_option('display.width', 180)
plt.rcParams['figure.figsize'] = (12, 4)

DB = Path(r'D:\\Projects\\done\\eth_bot\\eth_bot_v2') / 'database' / 'market.db'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

def load_close(symbol, timeframe='1d'):
    d = sql("SELECT open_time, close FROM ohlcv "
            "WHERE symbol=? AND timeframe=? ORDER BY open_time",
            symbol=symbol, timeframe=timeframe)
    d['date'] = pd.to_datetime(d.open_time, unit='ms', utc=True).tz_convert(None)
    return d.set_index('date')['close']


def ttest(series, name):
    if len(series) < 5:
        return {'name': name, 'mean': series.mean(), 't': np.nan, 'p': np.nan, 'n': len(series), 'sig': False}
    t, p = stats.ttest_1samp(series, 0)
    return {'name': name, 'mean': series.mean(), 't': float(t), 'p': float(p), 'n': len(series), 'sig': bool(p < 0.05)}


## 1. Load data and compute month

In [ ]:
data = {}
for sym in SYMBOLS:
    c = load_close(sym, '1d')
    df = pd.DataFrame({'close': c})
    df['ret'] = df['close'].pct_change()
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['quarter'] = df.index.quarter
    data[sym] = df.dropna()

for sym, df in data.items():
    print(f'{sym}: {len(df)} days  {df.index.min().date()} -> {df.index.max().date()}')

## 2. Mean return by month

In [ ]:
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

for sym, df in data.items():
    print(f'\n=== {sym} ===')
    stats_df = df.groupby('month')['ret'].agg(mean='mean', std='std', count='count')
    stats_df['mean_%'] = stats_df['mean'] * 100
    stats_df['std_%'] = stats_df['std'] * 100
    stats_df['hit_rate'] = df.groupby('month')['ret'].apply(lambda s: (s > 0).mean())
    stats_df.index = MONTHS
    print(stats_df[['mean_%', 'std_%', 'hit_rate', 'count']].round(4).to_string())

## 3. Bar chart - mean return by month

**What this tells you:** which months are historically profitable. But remember: **6 years of data = only 6 samples per month**. Small samples can be very misleading.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for ax, sym in zip(axes, SYMBOLS):
    df = data[sym]
    means = df.groupby('month')['ret'].mean() * 100
    colors = ['seagreen' if v > 0 else 'crimson' for v in means.values]
    bars = ax.bar(MONTHS, means.values, color=colors)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{sym} - mean daily return by month')
    ax.set_ylabel('mean daily return %')
    for b, v in zip(bars, means.values):
        ax.text(b.get_x() + b.get_width()/2, v + (0.005 if v >= 0 else -0.01),
                f'{v:+.3f}', ha='center', fontsize=8)

plt.suptitle('Sell in May? Uptober? Rektember? Test with data.', fontsize=13)
plt.tight_layout(); plt.show()

## 4. Volatility by month

**What this tells you:** some months are consistently volatile. If you're buying options, cheap premium months are your friend.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for ax, sym in zip(axes, SYMBOLS):
    df = data[sym]
    stds = df.groupby('month')['ret'].std() * 100
    colors = plt.cm.viridis(stds / stds.max())
    ax.bar(MONTHS, stds.values, color=colors)
    ax.set_title(f'{sym} - volatility by month')
    ax.set_ylabel('std of daily return %')

plt.suptitle('Which months are more volatile?', fontsize=13)
plt.tight_layout(); plt.show()

## 5. Year × month heatmap

**The critical test:** does the pattern repeat across years?

- Green cells = that month was up in that year
- Red cells = that month was down
- **Consistent column** (e.g., all years green in October) = real pattern
- **Random mix** = no pattern

In [ ]:
for sym in SYMBOLS:
    df = data[sym].copy()
    df['ym_year'] = df.index.year
    df['ym_month'] = df.index.month

    monthly = (1 + df['ret']).groupby([df['ym_year'], df['ym_month']]).prod() - 1
    pivot = monthly.unstack('ym_month') * 100
    pivot.columns = MONTHS

    fig, ax = plt.subplots(figsize=(13, 5))
    im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=-15, vmax=15, aspect='auto')
    ax.set_xticks(range(len(MONTHS)))
    ax.set_xticklabels(MONTHS)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.iloc[i, j]
            if pd.notna(v):
                ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                        fontsize=9, color='white' if abs(v) > 8 else 'black')
    plt.colorbar(im, ax=ax, label='monthly return %')
    ax.set_title(f'{sym} - monthly returns by year (%)')
    plt.tight_layout(); plt.show()

## 6. Quarter analysis

Grouping months into quarters - does Q4 outperform Q1?

In [ ]:
QUARTERS = ['Q1 (Jan-Mar)', 'Q2 (Apr-Jun)', 'Q3 (Jul-Sep)', 'Q4 (Oct-Dec)']

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for ax, sym in zip(axes, SYMBOLS):
    df = data[sym]
    q_means = df.groupby('quarter')['ret'].mean() * 100
    colors = ['seagreen' if v > 0 else 'crimson' for v in q_means.values]
    ax.bar(QUARTERS, q_means.values, color=colors)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{sym} - mean return by quarter')
    ax.set_ylabel('mean daily return %')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Does Q4 actually outperform?', fontsize=13)
plt.tight_layout(); plt.show()

## 7. Statistical significance

With only ~6 years of data, each month has ~180 observations. Enough for a t-test but low power. Be skeptical.

In [ ]:
results = []
for sym, df in data.items():
    for m in range(1, 13):
        r = df[df.month == m]['ret']
        res = ttest(r, MONTHS[m-1])
        res['symbol'] = sym
        results.append(res)

res_df = pd.DataFrame(results)
res_df['mean_%'] = (res_df['mean'] * 100).round(4)
res_df['p'] = res_df['p'].round(4)
print(res_df[['symbol', 'name', 'mean_%', 'p', 'n', 'sig']].to_string(index=False))

## 8. Verdict

In [ ]:
print('=' * 72)
print('VERDICT - MONTH OF YEAR SEASONALITY')
print('=' * 72)
print()

for sym in SYMBOLS:
    s = res_df[res_df.symbol == sym]
    sig = s[s.sig]

    if len(sig) > 0:
        print(f'{sym}: STATISTICALLY SIGNIFICANT MONTHS:')
        for _, r in sig.iterrows():
            print(f"  {r['name']}: {r['mean_%']:+.4f}% (p={r['p']:.4f})")
    else:
        print(f'{sym}: No month is statistically significant')
        best = s.loc[s['mean'].idxmax()]
        worst = s.loc[s['mean'].idxmin()]
        print(f"  Best  : {best['name']} at {best['mean_%']:+.4f}%")
        print(f"  Worst : {worst['name']} at {worst['mean_%']:+.4f}%")
        print(f"  Neither is significant (p > 0.05)")

print()
print('=' * 72)
print('INTERPRETATION')
print('=' * 72)
print()
print('What folklore claims:')
print('  - "Sell in May" - summer weak')
print('  - "Uptober" - October bullish')
print('  - "Rektember" - September bearish')
print()
print('What the data usually shows:')
print('  - 6 years = ~180 observations per month')
print('  - High variance between years')
print('  - Rarely statistically significant')
print('  - The patterns you see are usually 2-3 outlier years dominating')
print()
print('Use this for:')
print('  - Awareness of where big months historically clustered')
print('  - NOT for calendar-based entries')
print()
print('Better signals exist in:')
print('  - Option PCR and OI positioning')
print('  - Funding rates and open interest')
print('  - Momentum and volatility regimes')

## Final note

Month-of-year effects in crypto are **mostly narrative**. A few extreme months (March 2020 crash, November 2021 ATH, etc.) skew the averages.

**If you must trade seasonality**, use it as one input among many. Never enter a position just because 'it's October'.